<a href="https://colab.research.google.com/github/SinemKar/TurkeyEarthquake/blob/main/SinemCerit_DepremAnalizi_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Türkiye Deprem Verilerinin Görselleştirme Analizi (1994–2023)

**Hazırlayan:** Sinem Cerit  
**Öğrenci Numarası:** N25210609  
**Kurum:** Hacettepe Üniversitesi, Veri ve Bilgi Mühendisliği  
**E-posta:** sinemcerit26@hacettepe.edu.tr  
**Colab Linki:** [Google Colab](https://colab.research.google.com/drive/1SpRud5duVpIAD1oP5mj1g0diTpDRkSJx?usp=sharing)


---

## Abstract

Bu çalışmada Türkiye ve çevresinde 1994–2023 yılları arasında kaydedilen yaklaşık 50.000 deprem olayı incelenmiştir. Kaggle platformundan elde edilen veri seti; zamansal, mekânsal ve etkileşimli görselleştirme teknikleriyle analiz edilmiştir. Yıllık deprem frekansı ve magnitüd eğilimleri çubuk ve çizgi grafiklerle, coğrafi dağılım nokta haritasıyla, saatlik dağılım ise etkileşimli polar bar grafikle sunulmuştur. Bulgular; KAF ve DAF fay hatları boyunca belirgin uzamsal kümelenmeler ile 2023 Kahramanmaraş depreminin tarihsel açıdan istisnai niteliğini ortaya koymaktadır.

**Index Terms:** deprem görselleştirme, zaman serisi analizi, coğrafi haritalama, etkileşimli görselleştirme, Türkiye sismik aktivitesi


---

## 1. Giriş

Türkiye, Kuzey Anadolu Fay Hattı (KAF) ve Doğu Anadolu Fay Hattı (DAF) üzerinde yer alması nedeniyle dünyanın en yüksek sismik aktiviteye sahip bölgelerinden birini barındırmaktadır. Bu çalışmada 1994–2023 dönemine ait yaklaşık 50.000 kayıt içeren veri seti görselleştirme yöntemleriyle incelenmiştir.

Çalışma üç temel araştırma sorusu etrafında yapılandırılmıştır: (1) Yıllık deprem frekansı ve büyüklüğü nasıl değişmiştir? (2) Depremler coğrafi olarak nerede kümelenmektedir? (3) Saatlik dağılım magnitüd kategorisine göre farklılaşmakta mıdır?


---

## 2. Veri Seti

Analizde Kaggle platformundan elde edilen "Turkey Earthquake Data (1994–2023)" veri seti kullanılmıştır. Ham veri seti 50.000 satır ve 15 sütundan oluşmakta olup ön işleme (geçersiz magnitüd ve derinlik değerlerinin çıkarılması) sonrasında 21.246 kayıt analize dahil edilmiştir. Veri seti; oluşum tarihi, saati, enlem, boylam, odak derinliği, yerel magnitüd (ML), moment magnitüdü (Mw) ve yer bilgisini kapsamaktadır.


---

## 📚 Temel Terminoloji ve Kavram Sözlüğü

Rapor içerisindeki analizlerin daha net anlaşılabilmesi adına öne çıkan temel kavramların açıklamaları aşağıda özetlenmiştir:

- **Magnitüd (ML / Mw / Ms):** Depremin açığa çıkardığı enerjiyi ölçen logaritmik skala. Bu çalışmada öncelikle Yerel Magnitüd (ML) kullanılmış; eksik değerlerde sırasıyla Mw ve Ms ile tamamlanmıştır.
- **KAF (Kuzey Anadolu Fay Hattı):** Marmara'dan Erzincan'a uzanan, dünyanın en aktif doğrultu atımlı faylarından biri. 1999 Gölcük (Mw 7.6) ve Düzce (Mw 7.2) depremleri bu hat üzerinde gerçekleşmiştir.
- **DAF (Doğu Anadolu Fay Hattı):** Kahramanmaraş'tan başlayarak kuzeydoğuya uzanan fay hattı; 2023 Kahramanmaraş deprem çiftinin (Mw 7.7 + Mw 7.6) kaynağıdır.
- **Artçı Şok:** Ana depremden sonra aynı bölgede gerçekleşen, genellikle daha küçük büyüklüklü sarsıntılar. 2023 yılındaki yüksek deprem sayısının temel nedenidir.
- **Odak Derinliği (Der):** Depremin yer altındaki başlangıç noktasının yüzeye olan mesafesi (km). Sığ depremler (< 70 km) genellikle daha fazla hasar yaratır.
- **Polar Bar Chart (Gül Diyagramı):** Döngüsel verilerin (saat, ay, yön gibi) görselleştirilmesinde kullanılan, her dilimin bir kategoriyi ve uzunluğunun frekansı temsil ettiği grafik türü.


---

## ⚙️ Veri Yükleme ve Ön İşleme


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings, os
warnings.filterwarnings('ignore')

os.makedirs('gorseller', exist_ok=True)
plt.rcParams['figure.dpi'] = 120

url = 'https://raw.githubusercontent.com/SinemKar/TurkeyEarthquake/refs/heads/main/TurkeyEarthquake.csv'
df = pd.read_csv(url, sep=',', encoding='utf-8', skipinitialspace=True)
df.columns = df.columns.str.strip()

df['Tarih'] = pd.to_datetime(df['Olus tarihi'], format='%Y.%m.%d', errors='coerce')
df['Yil']   = df['Tarih'].dt.year

df['Mag'] = df['ML'].replace(0, np.nan)
df.loc[df['Mag'].isna(), 'Mag'] = df.loc[df['Mag'].isna(), 'Mw'].replace(0, np.nan)
df.loc[df['Mag'].isna(), 'Mag'] = df.loc[df['Mag'].isna(), 'Ms'].replace(0, np.nan)

df = df.dropna(subset=['Tarih', 'Mag'])
df = df[df['Mag'] > 0]
df = df[df['Der(km)'] > 0]

print(f'Kayıt sayısı : {len(df):,}')
print(f'Tarih aralığı: {df["Tarih"].min().date()} → {df["Tarih"].max().date()}')


---

## 3. Zamansal Analiz

### 3.1 Araştırma Sorusu

Türkiye'de yıllık deprem sayısı ve ortalama deprem büyüklükleri 1994–2023 arasında nasıl değişmiştir?

### 3.2 Görselleştirme ve Bulgular

1994–2022 döneminde yıllık deprem sayısı genellikle 500–2.000 aralığında seyretmiştir. 2023 yılı, Şubat ayındaki Kahramanmaraş deprem çifti (Mw 7.7 + Mw 7.6) ve uzun süren artçı şoklar nedeniyle bu tarihin en yüksek yıllık aktivitesini kaydetmiştir. Ortalama deprem büyüklüğü yıllar içinde stabil bir seyir izlerken, maksimum büyüklük belirli yıllarda dikkat çekici artışlar göstermektedir.

İki panel birlikte okunduğunda deprem sayısındaki artışın büyük ölçüde artçı şoklardan kaynaklandığı, ortalama şiddetin ise değişmediği anlaşılmaktadır.


**TABLO I. Soru 1 — Veri Tipleri**

| **Değişken** | **Veri Tipi** |
| --- | --- |
| Yıl | Sıralı |
| Deprem Sayısı | Nicel |
| Ortalama Deprem Büyüklüğü | Nicel |
| Maksimum Deprem Büyüklüğü | Nicel |
| Önemli Deprem Olayları | Kategorik |


**TABLO II. Yıllık Toplam Deprem Sayısı — Katalog Bilgisi**

| **Katalog Öğesi** | **Açıklama** |
| --- | --- |
| Grafik türü | Çubuk grafiği |
| X ekseni | Yıl |
| Y ekseni | Deprem sayısı |
| İşaretleyici | Çubuk |
| Renk kanalı | Sarıdan kırmızıya |
| Ek işaretleyici | Kesikli trend çizgisi |


**TABLO III. Yıllık Ortalama ve Maksimum Büyüklük — Katalog Bilgisi**

| **Katalog Öğesi** | **Açıklama** |
| --- | --- |
| Grafik türü | Çizgi grafik |
| X ekseni | Yıl |
| Y ekseni | Deprem büyüklüğü |
| Mavi çizgi | Ortalama büyüklük |
| Kırmızı kesikli | Maksimum büyüklük |
| Turuncu noktalı | M = 5 eşik değeri |
| Alan dolgusu | Ortalama eğrisinin altında açık mavi |


In [ ]:
# Soru 1 — Zamansal Analiz
yillik = df.groupby('Yil').agg(
    Deprem_Sayisi=('Mag', 'count'),
    Ort_Mag=('Mag', 'mean'),
    Max_Mag=('Mag', 'max')
).reset_index()

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(11, 8),
    sharex=True,
    gridspec_kw={'hspace': 0.12, 'height_ratios': [1.4, 1]}
)
fig.suptitle('Türkiye Deprem Aktivitesi (1994–2023)',
             fontsize=16, fontweight='bold', y=1.01)

norm   = yillik['Deprem_Sayisi'] / yillik['Deprem_Sayisi'].max()
colors = plt.cm.YlOrRd(norm)

ax1.bar(yillik['Yil'], yillik['Deprem_Sayisi'],
        color=colors, edgecolor='white', linewidth=0.4, width=0.75)
ax1.plot(yillik['Yil'], yillik['Deprem_Sayisi'],
         'k--', alpha=0.35, linewidth=1.2, label='Trend')

olaylar = {
    1999: ('1999 Marmara\n(Mw 7.6)', 'darkred',   -4, 300),
    2020: ('2020 İzmir\n(Mw 7.0)',   'darkorange',  2, 400),
    2023: ('2023 Kahramanmaraş\n(Mw 7.7)', 'crimson', -4, 600),
}
for yil, (metin, renk, dx, dy) in olaylar.items():
    if yil in yillik['Yil'].values:
        val = yillik.loc[yillik['Yil'] == yil, 'Deprem_Sayisi'].values[0]
        ax1.annotate(metin, xy=(yil, val), xytext=(yil+dx, val+dy),
                     fontsize=8.5, color=renk, ha='center',
                     arrowprops=dict(arrowstyle='->', color=renk, lw=1.3))

ax1.set_ylabel('Deprem Sayısı', fontsize=11)
ax1.set_title('Yıllık Toplam Deprem Sayısı', fontsize=12, pad=8)
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.legend(fontsize=9)
ax1.set_ylim(0)

ax2.fill_between(yillik['Yil'], yillik['Ort_Mag'], alpha=0.2, color='steelblue')
ax2.plot(yillik['Yil'], yillik['Ort_Mag'],
         'o-', color='steelblue', lw=2, ms=5, label='Ort. Deprem Büyüklüğü')
ax2.plot(yillik['Yil'], yillik['Max_Mag'],
         's--', color='crimson', lw=1.8, ms=5, label='Maks. Deprem Büyüklüğü')
ax2.axhline(5, color='orange', linestyle=':', lw=1.5, label='M = 5 eşiği')

ax2.set_xlabel('Yıl', fontsize=11)
ax2.set_ylabel('Deprem Büyüklüğü', fontsize=11)
ax2.set_title('Yıllık Ortalama ve Maksimum Deprem Büyüklüğü', fontsize=12, pad=8)
ax2.legend(fontsize=9, loc='upper left')
ax2.set_ylim(0)

plt.tight_layout()
plt.savefig('gorseller/soru1_zamansal.png', bbox_inches='tight')
plt.show()


*Fig. 1. Türkiye Deprem Aktivitesi (1994–2023). Üst panel: yıllık toplam deprem sayısı; alt panel: ortalama ve maksimum magnitüd.*


---

## 4. Mekânsal / Coğrafi Analiz

### 4.1 Araştırma Sorusu

Türkiye'de M ≥ 3 depremlerin coğrafi dağılımı nasıldır? Hangi bölgeler ve fay hatları öne çıkmaktadır?

### 4.2 Görselleştirme ve Bulgular

Depremler Türkiye genelinde homojen dağılmamaktadır; iki belirgin kümelenme ekseni öne çıkmaktadır.

**Kuzey Anadolu Fay Hattı (KAF):** Marmara'dan Erzincan'a uzanan şerit boyunca yoğun nokta kümesi oluşmaktadır. 1999 Gölcük (Mw 7.6) ve Düzce (Mw 7.2) depremleri bu hat üzerinde gerçekleşmiştir.

**Doğu Anadolu Fay Hattı (DAF):** Kahramanmaraş–Adıyaman hattı boyunca belirgin aktivite gözlemlenmektedir. 2023 deprem çifti (Mw 7.7 + Mw 7.6) bu fayın tarihsel kırılmasını temsil etmektedir.

**Ege Bölgesi:** İzmir ve Ege açıkları üçüncü önemli odak noktasını oluşturmaktadır. İç Anadolu ise görece düşük aktivite sergilemektedir.


**TABLO IV. Soru 2 — Veri Tipleri**

| **Değişken** | **Veri Tipi** |
| --- | --- |
| Boylam | Nicel |
| Enlem | Nicel |
| Magnitüd (Mag) | Nicel |
| Magnitüd Kategorisi | Sıralı |
| Şehir Adları | Kategorik |
| Fay Hattı Türü | Kategorik |


**TABLO V. Türkiye Deprem Dağılımı — Görselleştirme Yapısı**

| **Katalog Öğesi** | **Açıklama** |
| --- | --- |
| Grafik türü | Nokta haritası |
| Koordinat sistemi | Boylam × Enlem |
| İşaretleyici | Nokta |
| Boyut kanalı | Magnitüd kategorisi |
| Renk kanalı | mavi→turuncu→kırmızı→beyaz |
| Ek işaretleyici | Üçgen: şehir konumları; kesikli çizgi: fay hatları |


In [ ]:
# Soru 2 — Nokta Haritası
import matplotlib.patches as mpatches

df_map = df[df['Mag'] >= 3].copy()
bins   = [3, 4, 5, 6, 10]
labels = ['3–4 (Hafif)', '4–5 (Orta)', '5–6 (Güçlü)', '6+ (Büyük)']
df_map['MagKat'] = pd.cut(df_map['Mag'], bins=bins, labels=labels)

renk_harita = {
    '3–4 (Hafif)': ('#4fc3f7', 4,   0.15),
    '4–5 (Orta)':  ('#ffb74d', 18,  0.45),
    '5–6 (Güçlü)': ('#ef5350', 60,  0.75),
    '6+ (Büyük)':  ('#ffffff', 150, 1.00),
}

fig, ax = plt.subplots(figsize=(15, 8), facecolor='#1a1a2e')
ax.set_facecolor('#1a1a2e')

for kat in labels:
    alt = df_map[df_map['MagKat'] == kat]
    renk, boyut, alpha = renk_harita[kat]
    ax.scatter(alt['Boylam'], alt['Enlem'],
               s=boyut, color=renk, alpha=alpha,
               linewidths=0, label=f'{kat}  (n={len(alt):,})')

sehirler = {
    'İstanbul':      (28.97, 41.01),
    'Ankara':        (32.86, 39.93),
    'İzmir':         (27.14, 38.42),
    'Kahramanmaraş': (36.92, 37.57),
    'Erzincan':      (39.49, 39.75),
    'Düzce':         (31.15, 40.84),
}
for sehir, (lon, lat) in sehirler.items():
    ax.plot(lon, lat, '^', color='yellow', ms=5, alpha=0.9, zorder=5)
    ax.text(lon+0.15, lat+0.12, sehir, color='yellow', fontsize=7.5, alpha=0.9, zorder=5)

kaf_lon = [27, 29, 31, 33, 35, 37, 39, 41]
kaf_lat = [40.8, 40.6, 40.4, 40.1, 39.9, 39.7, 39.5, 39.6]
ax.plot(kaf_lon, kaf_lat, '--', color='#f9ca24', lw=1.8, alpha=0.7, label='KAF (yaklaşık)', zorder=4)

daf_lon = [36.1, 36.8, 37.5, 38.3, 39.0]
daf_lat = [37.2, 37.6, 38.0, 38.6, 39.3]
ax.plot(daf_lon, daf_lat, '--', color='#6ab04c', lw=1.8, alpha=0.7, label='DAF (yaklaşık)', zorder=4)

ax.set_xlim(25.5, 44.5)
ax.set_ylim(35.5, 43.0)
ax.set_title(
    'Türkiye Deprem Dağılımı — Dot Map (M ≥ 3, 1994–2023)\n'
    'Her nokta bir deprem olayını temsil eder; boyut ve renk magnitüdü kodlar',
    color='white', fontsize=13, pad=10
)
ax.set_xlabel('Boylam (°E)', color='white', fontsize=10)
ax.set_ylabel('Enlem (°N)', color='white', fontsize=10)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#444')

leg = ax.legend(loc='lower right', facecolor='#222',
                labelcolor='white', fontsize=8,
                title='Magnitüd Kategorisi / Fay Hattı', title_fontsize=8)
leg.get_title().set_color('lightgray')

plt.tight_layout()
plt.savefig('gorseller/soru2_dot_map.png', bbox_inches='tight', facecolor='#1a1a2e')
plt.show()


*Fig. 2. Türkiye Deprem Dağılımı — Dot Map (M ≥ 3, 1994–2023). Her nokta bir deprem olayını temsil etmekte; boyut ve renk magnitüdü kodlamaktadır. Sarı kesikli çizgi KAF, yeşil kesikli çizgi DAF fay hattını göstermektedir.*


---

## 5. Etkileşimli Görselleştirme

### 5.1 Araştırma Sorusu

Türkiye'de depremler günün hangi saatlerinde daha sık kaydedilmektedir? Saatlik dağılım magnitüd kategorisine göre farklılaşmakta mıdır?

### 5.2 Görselleştirme ve Bulgular

Saatlik dağılım incelendiğinde depremlerin günün tüm saatlerinde meydana geldiği ve belirli bir saat aralığında kesin bir yoğunlaşma olmadığı görülmektedir. Küçük ve orta büyüklükteki depremler dengeli bir dağılım gösterirken, M 4–5 ve M ≥ 5 kategorilerinde saatlik dalgalanmalar daha belirgin hale gelmektedir.

Büyük depremlerin veri setindeki sayısı görece az olduğundan bazı saatlerde yoğunlaşma varmış gibi görünen durumlar dikkatli yorumlanmalıdır. Saatlik toplam dağılım üzerinde en etkili grubun M 3–4 büyüklüğündeki depremler olduğu gözlemlenmiştir.


**TABLO VI. Soru 3 — Veri Tipleri**

| **Değişken** | **Veri Tipi** |
| --- | --- |
| Saat | Sıralı, döngüsel |
| Deprem Sayısı | Nicel |
| Magnitüd | Nicel |
| Magnitüd Kategorisi | Kategorik |


**TABLO VII. Saatlik Deprem Dağılımı — Görselleştirme Yapısı**

| **Katalog Öğesi** | **Açıklama** |
| --- | --- |
| Grafik türü | Polar bar chart |
| Açı ekseni | Saat yönünde |
| Yarıçap kanalı | Deprem sayısı |
| Renk kanalı | mavi→turuncu→kırmızı→beyaz |
| Filtre butonu | Tümü / M<3 / M3-4 / M4-5 / M≥5 |


In [ ]:
# Soru 3 — Etkileşimli Polar Bar Chart
import plotly.graph_objects as go

df['Saat'] = df['Olus zamani'].str[:2].astype(float, errors='ignore')
df_saat = df.dropna(subset=['Saat']).copy()
df_saat['Saat'] = pd.to_numeric(df_saat['Saat'], errors='coerce')
df_saat = df_saat.dropna(subset=['Saat'])
df_saat['Saat'] = df_saat['Saat'].astype(int)
df_saat = df_saat[df_saat['Saat'].between(0, 23)]

bins   = [0, 3, 4, 5, 10]
labels = ['M < 3', 'M 3–4', 'M 4–5', 'M ≥ 5']
df_saat['MagKat'] = pd.cut(df_saat['Mag'], bins=bins, labels=labels)

saat_etiketleri = [f'{s:02d}:00' for s in range(24)]
renkler_kat = {
    'M < 3':  '#4fc3f7',
    'M 3–4':  '#ffb74d',
    'M 4–5':  '#ef5350',
    'M ≥ 5':  '#ffffff',
}

fig = go.Figure()

for kat in labels:
    alt = df_saat[df_saat['MagKat'] == kat]
    sayim = alt.groupby('Saat').size().reindex(range(24), fill_value=0)
    hover = [
        f'<b>Saat {s:02d}:00</b><br>Deprem sayısı: {sayim[s]:,}<br>Kategori: {kat}'
        for s in range(24)
    ]
    fig.add_trace(go.Barpolar(
        r=sayim.values,
        theta=saat_etiketleri,
        name=kat,
        marker_color=renkler_kat[kat],
        marker_line_color='#1a1a2e',
        marker_line_width=0.5,
        opacity=0.85,
        hovertext=hover,
        hoverinfo='text',
        visible=True
    ))

n = len(labels)
goster_hepsi  = [True] * n
goster_birden = [[i == j for i in range(n)] for j in range(n)]

butonlar = [
    dict(label='Tümü', method='update',
         args=[{'visible': goster_hepsi},
               {'title.text': 'Saatlik Deprem Dağılımı — Tüm Magnitüdler'}])
]
for j, kat in enumerate(labels):
    butonlar.append(dict(
        label=kat, method='update',
        args=[{'visible': goster_birden[j]},
              {'title.text': f'Saatlik Deprem Dağılımı — {kat}'}]
    ))

fig.update_layout(
    title=dict(
        text='Saatlik Deprem Dağılımı — Türkiye (1994–2023)<br>'
             '<sup>Her dilim bir saati, uzunluk deprem sayısını gösterir</sup>',
        font=dict(color='white', size=13),
        x=0.5, y=0.97
    ),
    height=650,
    paper_bgcolor='#0d1117',
    font_color='white',
    polar=dict(
        bgcolor='#1a1a2e',
        angularaxis=dict(
            tickmode='array',
            tickvals=saat_etiketleri,
            ticktext=[f'{s:02d}:00' for s in range(24)],
            direction='clockwise',
            rotation=90,
            tickfont=dict(color='white', size=9),
            linecolor='#444',
            gridcolor='#2a2a3e',
        ),
        radialaxis=dict(
            tickfont=dict(color='white', size=10),
            linecolor='#444',
            gridcolor='#555555',
            showticklabels=True,
            layer='above traces',
        )
    ),
    legend=dict(
        font=dict(color='white'),
        bgcolor='#1a1a2e',
        bordercolor='#444',
        orientation='h',
        y=-0.12, x=0.5, xanchor='center'
    ),
    updatemenus=[dict(
        type='buttons',
        direction='right',
        showactive=True,
        x=0.5, xanchor='center',
        y=1.12, yanchor='top',
        buttons=butonlar,
        bgcolor='#2d3436',
        bordercolor='#636e72',
        font=dict(color='white', size=10),
        active=0
    )]
)

fig.write_html('gorseller/soru3_polar_saat.html')
fig.show()


*Fig. 3. Saatlik Deprem Dağılımı — Polar Bar Chart (1994–2023). Her dilim bir saati, yarıçap deprem sayısını kodlamaktadır. Üstteki butonlar ile magnitüd kategorisine göre filtreleme yapılabilmektedir.*


---

## 6. Sonuç

Bu çalışmada 1994–2023 yılları arasında Türkiye'de kaydedilen yaklaşık 50.000 deprem olayı üç farklı görselleştirme perspektifinden incelenmiştir. Zamansal analiz, 2023 yılının Kahramanmaraş artçı şokları nedeniyle tarihsel ortalamanın çok üzerinde seyrettiğini ortaya koymuştur. Mekânsal analiz, KAF ve DAF fay hatları boyunca belirgin kümelenmeleri görünür kılmıştır. Etkileşimli polar grafik ise deprem oluşumunun belirli bir saate bağlı olmadığını, ancak küçük magnitüd önyargısının gece saatlerinde gözlemlendiğini göstermiştir.

---

## Acknowledgments

Bu çalışma Hacettepe Üniversitesi Veri Görselleştirme dersi kapsamında hazırlanmıştır. Veri seti Kaggle platformundan elde edilmiştir.

---

## References

[1] C. Wilke, *Fundamentals of Data Visualization*, O'Reilly Media, 2019.  
[2] T. Munzner, *Visualization Analysis and Design*, CRC Press, 2015.  
[3] J. Heer, M. Bostock, and V. Ogievetsky, "A tour through the visualization zoo," *Communications of the ACM*, vol. 53, no. 6, pp. 59-67, 2010.  
[4] Turkey Earthquake Data (1994–2023), Kaggle, [Online]. Available: https://www.kaggle.com  

---
*Yapay Zekâ Kullanım Notu: Etkileşimli görselin kod yapısının geliştirilmesi ve açıklama metinlerinin düzenlenmesi aşamasında ChatGPT'den destek alınmıştır.*
